
# Triton Kernel für Matrix-Multiplikation
https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html

In [7]:
import numpy as np
import torch
import time
from TrappedAtomsSimulation.triton_matmul_kernels import tutorial_matmul, alternative_matmul

In [8]:
torch.set_float32_matmul_precision('high')



## Matrix dimensions
M = N = K = 2**10

dtype = torch.float32
device_gpu = 'cuda'
device_cpu = 'cpu'

N_REPEATS_GPU = N_REPEATS_CPU = 10

print(f"Benchmark MatMul: M={M}, N={N}, K={K}, dtype={dtype}")
print(f"Wiederholungen: {N_REPEATS_CPU}\n")

a_gpu = torch.randn(M, K, device=device_gpu, dtype=dtype).contiguous()
b_gpu = torch.randn(K, N, device=device_gpu, dtype=dtype).contiguous()
a_cpu = a_gpu.to(device_cpu)
b_cpu = b_gpu.to(device_cpu)

a_np = a_cpu.numpy()
b_np = b_cpu.numpy()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)


print("Berechnung mit NumPy (CPU)")
c_np = np.zeros((M, N), dtype=a_np.dtype)

start_time_np = time.perf_counter()
for _ in range(N_REPEATS_CPU):
    c_np = np.dot(a_np, b_np)
end_time_np = time.perf_counter()
np_time_ms = ((end_time_np - start_time_np) * 1000) / N_REPEATS_CPU
print("fertig")


print("Berechnung mit PyTorch (GPU)")
c_torch = torch.empty(M, N, device=device_gpu, dtype=dtype)

start_event.record()
for _ in range(N_REPEATS_GPU):
    c_torch = torch.matmul(a_gpu, b_gpu)
end_event.record()
torch.cuda.synchronize()
torch_time_ms = start_event.elapsed_time(end_event) / N_REPEATS_GPU
print("fertig")


print("Berechnung mit Triton-Kernel aus triton-Tutorial (GPU)")
c_triton = torch.empty(M, N, device=device_gpu, dtype=dtype)

for _ in range(10):
    c_triton = tutorial_matmul(a_gpu, b_gpu)
torch.cuda.synchronize()

start_event.record()
for _ in range(N_REPEATS_GPU):
    c_tutorial_triton = tutorial_matmul(a_gpu, b_gpu)
end_event.record()
torch.cuda.synchronize()
tutorial_triton_time_ms = start_event.elapsed_time(end_event) / N_REPEATS_GPU
print("fertig")

print("Berechnung mit alternativem Triton-Kernel (GPU)")
c_triton = torch.empty(M, N, device=device_gpu, dtype=dtype)

for _ in range(10):
    c_triton = alternative_matmul(a_gpu, b_gpu)
torch.cuda.synchronize()

start_event.record()
for _ in range(N_REPEATS_GPU):
    c_triton = alternative_matmul(a_gpu, b_gpu)
end_event.record()
torch.cuda.synchronize()
alternative_triton_time_ms = start_event.elapsed_time(end_event) / N_REPEATS_GPU
print("fertig")


print("\nKorrektheitsprüfung: Numpy als Referenz")
try:
    np.allclose(c_np, c_torch.cpu().numpy(), atol=1e-5, rtol=1e-4)
    print("Berechnung mit PyTorch korrekt")

    np.allclose(c_np, c_tutorial_triton.cpu().numpy(), atol=1e-5, rtol=1e-4)
    print("Berechnung mit Triton-Kernel aus triton-Tutorial korrekt")

    np.allclose(c_np, c_triton.cpu().numpy(), atol=1e-5, rtol=1e-4)
    print("Berechnung mit alternativem Triton-Kernel korrekt")

except Exception as e:
    print(f"  Inkorrekte Berechnung. Fehler: {e}")

print("\n--- Benchmark-Ergebnisse (MatMul) ---")
print(f"NumPy (CPU):\t\t\t\t\t {np_time_ms:.4f} ms")
print(f"PyTorch (GPU, cuBLAS):\t\t\t\t {torch_time_ms:.4f} ms")
print(f"Triton-Kernel aus Triton-Tutorial (GPU):\t {tutorial_triton_time_ms:.4f} ms")
print(f"Alternative Triton-Kernel (GPU):\t\t {alternative_triton_time_ms:.4f} ms")

Benchmark MatMul: M=1024, N=1024, K=1024, dtype=torch.float32
Wiederholungen: 10

Berechnung mit NumPy (CPU)
fertig
Berechnung mit PyTorch (GPU)
fertig
Berechnung mit Triton-Kernel aus triton-Tutorial (GPU)
fertig
Berechnung mit alternativem Triton-Kernel (GPU)
fertig

Korrektheitsprüfung: Numpy als Referenz
Berechnung mit PyTorch korrekt
Berechnung mit Triton-Kernel aus triton-Tutorial korrekt
Berechnung mit alternativem Triton-Kernel korrekt

--- Benchmark-Ergebnisse (MatMul) ---
NumPy (CPU):					 2.8473 ms
PyTorch (GPU, cuBLAS):				 0.0689 ms
Triton-Kernel aus Triton-Tutorial (GPU):	 0.1082 ms
Alternative Triton-Kernel (GPU):		 0.1074 ms


In [9]:
(870/2.8)**(1/3)

6.773093544285512